<a href="https://colab.research.google.com/github/akshatharcs24-ship-it/Ai-lab/blob/main/1WA24CS033_Week9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

# Step 1: Eliminate biconditionals and implications
def eliminate_implications(expr):
    expr = re.sub(
        r'(\w+)\s*<=>\s*(\w+)',
        r'((\1 => \2) & (\2 => \1))',
        expr
    )

    expr = re.sub(
        r'(\w+)\s*=>\s*(\w+)',
        r'(~\1 | \2)',
        expr
    )
    return expr


# Step 2: Move negations inward
def move_negations(expr):
    expr = expr.replace('~(~', '(')

    expr = re.sub(
        r'~\s*forall\s+(\w+)\s+',
        r'exists \1 ',
        expr
    )

    expr = re.sub(
        r'~\s*exists\s+(\w+)\s+',
        r'forall \1 ',
        expr
    )

    return expr


# Step 3: Standardize variables
def standardize_variables(expr):
    variables = re.findall(
        r'(forall|exists)\s+(\w+)', expr
    )

    used = set()

    for quantifier, var in variables:
        if var not in used:
            used.add(var)
        else:
            new_var = var + str(len(used))
            expr = re.sub(
                rf'\b{quantifier}\s+{var}\b',
                f'{quantifier} {new_var}',
                expr,
                count=1
            )
            expr = re.sub(
                rf'\b{var}\b',
                new_var,
                expr
            )
            used.add(new_var)

    return expr


# Step 4: Skolemization
def skolemize(expr):
    counter = 1

    # Replace existential variables with Skolem constants
    while re.search(r'exists\s+(\w+)', expr):
        var = re.search(
            r'exists\s+(\w+)', expr
        ).group(1)

        skolem = 'SK' + str(counter)
        counter += 1

        expr = re.sub(
            rf'\bexists\s+{var}\b', '', expr
        )
        expr = re.sub(
            rf'\b{var}\b', skolem, expr
        )

    return expr


# Step 5: Remove universal quantifiers
def remove_universal(expr):
    expr = re.sub(
        r'\bforall\s+\w+\s*', '', expr
    )
    return expr


# Step 6: Distribute OR over AND
def distribute_or(expr):
    expr = expr.strip()

    # Basic distribution:
    # A | (B & C) = (A | B) & (A | C)

    pattern = r'(\w+)\s*\|\s*\((\w+)\s*&\s*(\w+)\)'

    while re.search(pattern, expr):
        expr = re.sub(
            pattern,
            r'((\1 | \2) & (\1 | \3))',
            expr
        )

    return expr


# Main CNF conversion function
def convert_to_cnf(expr):
    print("\nOriginal Expression:", expr)

    expr = eliminate_implications(expr)
    print("1. Eliminate implications:", expr)

    expr = move_negations(expr)
    print("2. Move negations inward:", expr)

    expr = standardize_variables(expr)
    print("3. Standardize variables:", expr)

    expr = skolemize(expr)
    print("4. Skolemization:", expr)

    expr = remove_universal(expr)
    print("5. Remove universal quantifiers:", expr)

    expr = distribute_or(expr)
    print("6. Final CNF:", expr)

    return expr


# Main program
print("FOL to CNF Converter")
print("--------------------")

expression = input("Enter FOL expression: ")

result = convert_to_cnf(expression)

print("\nCNF Result:", result)


FOL to CNF Converter
--------------------
Enter FOL expression: forall x (P(x) => Q(x))

Original Expression: forall x (P(x) => Q(x))
1. Eliminate implications: forall x (P(x) => Q(x))
2. Move negations inward: forall x (P(x) => Q(x))
3. Standardize variables: forall x (P(x) => Q(x))
4. Skolemization: forall x (P(x) => Q(x))
5. Remove universal quantifiers: (P(x) => Q(x))
6. Final CNF: (P(x) => Q(x))

CNF Result: (P(x) => Q(x))
